In [7]:
import time
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [8]:
df = pd.read_csv('email_classification_dataset.csv')

In [9]:
df.shape

(10000, 3)

In [10]:
df.head()

,id,email,label
0,2685,From: support@legitcompany.com\nSubject: Regar...,ham
1,5857,From: noreply@softwareupdates.com\nSubject: We...,ham
2,2399,From: noreply@softwareupdates.com\nSubject: Im...,ham
3,3244,From: info@customerservice.co\nSubject: Team S...,ham
4,2844,From: info@customerservice.co\nSubject: Team S...,ham


###  Check label distribution (class balance)

In [11]:
print("Label distribution:")
print(df['label'].value_counts())

Label distribution:
label
ham     8500
spam    1500
Name: count, dtype: int64


##  Preprocessing

In [12]:
print("Missing values in each column:")
print(df.isnull().sum())

Missing values in each column:
id       0
email    0
label    0
dtype: int64


In [13]:
# Drop 'id' column as it is not a useful feature for classification
df = df.drop(columns=['id'])

In [14]:
# Encode categorical target variable 'label' ('ham'->0, 'spam'->1)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])
print("Encoded label classes:", le.classes_) 

Encoded label classes: ['ham' 'spam']


In [15]:
X = df['email']
y = df['label_encoded']

## Train-Test Split with Class Balancing

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

# Split data into training and test sets with stratification to preserve class distribution
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Training set size:", X_train_text.shape[0])
print("Test set size:", X_test_text.shape[0])

# Create a DataFrame for the training set to facilitate resampling
train_df = pd.DataFrame({'email': X_train_text, 'label': y_train})

# Separate majority and minority classes in the training data
df_majority = train_df[train_df.label == 0]  # ham (majority)
df_minority = train_df[train_df.label == 1]  # spam (minority)
print("Class distribution before resampling (train):")
print(train_df['label'].value_counts())

Training set size: 8000
Test set size: 2000
Class distribution before resampling (train):
label
0    6800
1    1200
Name: count, dtype: int64


In [17]:
# Oversample the minority class to match the majority class size
df_minority_upsampled = resample(
    df_minority,
    replace=True,          # sample with replacement
    n_samples=len(df_majority),  # match number of majority class samples
    random_state=42
)
# Combine majority class with upsampled minority class
df_train_balanced = pd.concat([df_majority, df_minority_upsampled])
# Shuffle the balanced dataset
df_train_balanced = df_train_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
print("Class distribution after resampling (train):")
print(df_train_balanced['label'].value_counts())

# Separate features and target from the balanced training set
X_train_balanced_text = df_train_balanced['email']
y_train_balanced = df_train_balanced['label']

Class distribution after resampling (train):
label
1    6800
0    6800
Name: count, dtype: int64


## Feature Ectraction

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Convert text emails into TF-IDF features
tfidf = TfidfVectorizer(stop_words='english')
X_train_vect = tfidf.fit_transform(X_train_balanced_text)
X_test_vect = tfidf.transform(X_test_text)
print("Number of TF-IDF features:", X_train_vect.shape[1])

Number of TF-IDF features: 252


# K-Nearest Neighbors

In [19]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

knn = KNeighborsClassifier()
knn.fit(X_train_vect, y_train_balanced)
y_pred_knn = knn.predict(X_test_vect)

acc_knn = accuracy_score(y_test, y_pred_knn)
prec_knn = precision_score(y_test, y_pred_knn)
rec_knn = recall_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)

print("KNN Performance:")
print(f"Accuracy: {acc_knn:.4f}, Precision: {prec_knn:.4f}, Recall: {rec_knn:.4f}, F1-Score: {f1_knn:.4f}")


KNN Performance:
Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1-Score: 1.0000


# SVM

In [20]:
from sklearn.svm import SVC

svm = SVC(kernel='linear', probability=True, random_state=42)
svm.fit(X_train_vect, y_train_balanced)
y_pred_svm = svm.predict(X_test_vect)

acc_svm = accuracy_score(y_test, y_pred_svm)
prec_svm = precision_score(y_test, y_pred_svm)
rec_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)

print("SVM Performance:")
print(f"Accuracy: {acc_svm:.4f}, Precision: {prec_svm:.4f}, Recall: {rec_svm:.4f}, F1-Score: {f1_svm:.4f}")


SVM Performance:
Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1-Score: 1.0000


# Decision Tree

In [21]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_vect, y_train_balanced)
y_pred_dt = dt.predict(X_test_vect)

acc_dt = accuracy_score(y_test, y_pred_dt)
prec_dt = precision_score(y_test, y_pred_dt)
rec_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)

print("Decision Tree Performance:")
print(f"Accuracy: {acc_dt:.4f}, Precision: {prec_dt:.4f}, Recall: {rec_dt:.4f}, F1-Score: {f1_dt:.4f}")


Decision Tree Performance:
Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1-Score: 1.0000


# Random Forest

In [22]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_vect, y_train_balanced)
y_pred_rf = rf.predict(X_test_vect)

acc_rf = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf)
rec_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print("Random Forest Performance:")
print(f"Accuracy: {acc_rf:.4f}, Precision: {prec_rf:.4f}, Recall: {rec_rf:.4f}, F1-Score: {f1_rf:.4f}")


Random Forest Performance:
Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1-Score: 1.0000


# AdaBoost

In [23]:
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(n_estimators=50, random_state=42)
ada.fit(X_train_vect, y_train_balanced)
y_pred_ada = ada.predict(X_test_vect)

acc_ada = accuracy_score(y_test, y_pred_ada)
prec_ada = precision_score(y_test, y_pred_ada)
rec_ada = recall_score(y_test, y_pred_ada)
f1_ada = f1_score(y_test, y_pred_ada)

print("AdaBoost Performance:")
print(f"Accuracy: {acc_ada:.4f}, Precision: {prec_ada:.4f}, Recall: {rec_ada:.4f}, F1-Score: {f1_ada:.4f}")


AdaBoost Performance:
Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1-Score: 1.0000


# Gradient Boosting

In [24]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train_vect, y_train_balanced)
y_pred_gb = gb.predict(X_test_vect)

acc_gb = accuracy_score(y_test, y_pred_gb)
prec_gb = precision_score(y_test, y_pred_gb)
rec_gb = recall_score(y_test, y_pred_gb)
f1_gb = f1_score(y_test, y_pred_gb)

print("Gradient Boosting Performance:")
print(f"Accuracy: {acc_gb:.4f}, Precision: {prec_gb:.4f}, Recall: {rec_gb:.4f}, F1-Score: {f1_gb:.4f}")


Gradient Boosting Performance:
Accuracy: 0.9995, Precision: 1.0000, Recall: 0.9967, F1-Score: 0.9983


# XGBoost

In [26]:
from xgboost import XGBClassifier

xgb = XGBClassifier(eval_metric='logloss', random_state=42)
xgb.fit(X_train_vect, y_train_balanced)
y_pred_xgb = xgb.predict(X_test_vect)

acc_xgb = accuracy_score(y_test, y_pred_xgb)
prec_xgb = precision_score(y_test, y_pred_xgb)
rec_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)

print("XGBoost Performance:")
print(f"Accuracy: {acc_xgb:.4f}, Precision: {prec_xgb:.4f}, Recall: {rec_xgb:.4f}, F1-Score: {f1_xgb:.4f}")


XGBoost Performance:
Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1-Score: 1.0000


# LightGBM

In [27]:
    from lightgbm import LGBMClassifier

    lgbm = LGBMClassifier(random_state=42)
    lgbm.fit(X_train_vect, y_train_balanced)
    y_pred_lgbm = lgbm.predict(X_test_vect)

    acc_lgbm = accuracy_score(y_test, y_pred_lgbm)
    prec_lgbm = precision_score(y_test, y_pred_lgbm)
    rec_lgbm = recall_score(y_test, y_pred_lgbm)
    f1_lgbm = f1_score(y_test, y_pred_lgbm)

    print("LightGBM Performance:")
    print(f"Accuracy: {acc_lgbm:.4f}, Precision: {prec_lgbm:.4f}, Recall: {rec_lgbm:.4f}, F1-Score: {f1_lgbm:.4f}")


[LightGBM] [Info] Number of positive: 6800, number of negative: 6800
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006523 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 30265
[LightGBM] [Info] Number of data points in the train set: 13600, number of used features: 252
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spli

C:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
